# 怎样把带标签的文件整理成可复现的数据集？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

label        0   1
split             
test         6   6
train       18  18
validation   6   6


## 1. 补全：数量守恒
验证三个划分合计 60 条，且每人恰有 2 条记录。

In [2]:
assert manifest.groupby('split').size().sum() == 60
assert manifest.groupby('person_id').size().eq(2).all()
print(manifest['split'].value_counts())

split
train         36
test          12
validation    12
Name: count, dtype: int64


## 2. 修改：换种子不破坏分组
用 random_state=19 重做按人 80/20 划分，验证人没有重叠。

In [3]:
left, right = train_test_split(people, test_size=0.2, stratify=people['label'], random_state=19)
assert set(left['person_id']).isdisjoint(right['person_id'])
assert len(right) == 6
print(right)

    person_id  label
50         25      1
30         15      1
52         26      0
54         27      1
20         10      0
8           4      0


## 3. 迁移：检测被修改的文件
对内存中的内容加一个字符后重算哈希，证明与清单不同；不要损坏原始文件。

In [4]:
row = manifest.iloc[0]
original = (CASE_DIR / row['path']).read_bytes()
changed_hash = hashlib.sha256(original + b'!').hexdigest()
assert changed_hash != row['sha256']
assert hashlib.sha256(original).hexdigest() == row['sha256']
print('内容变更检测通过')

内容变更检测通过
